# Validação preditiva do NECr

Testa se o NECr por lista (partido × UF) é um bom preditor do número de
cadeiras efetivamente conquistadas nas eleições de 2014, 2018 e 2022.

**Modelos estimados**

1. `S ~ NECr` — NECr sozinho
2. `S ~ B_prev` — bancada anterior (linha de base)
3. `S ~ B_prev + NECr + log(VR_total)` — modelo combinado

Onde:
- `S` = cadeiras conquistadas (partido × UF × ano)
- `NECr` = número efetivo de candidatos por recursos (Laakso-Taagepera)
- `B_prev` = bancada na eleição imediatamente anterior (mesmo partido × UF)
- `VR_total` = total de recursos de partido repassados à lista

**Validação cruzada**: estima em 2018, prediz 2022 (e vice-versa).

In [ ]:
import os
from pathlib import Path
ROOT = Path().resolve().parent
os.chdir(ROOT)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Cadeiras conquistadas (S)

In [ ]:
res = pd.read_parquet('data/processed/resultados.parquet')

ANOS_ALVO = [2014, 2018, 2022]

frames = []
for ano in ANOS_ALVO:
    r = res[
        (res.ano_eleicao == ano) &
        (res.ds_cargo == 'Deputado Federal') &
        (res.nr_turno == 1)
    ].drop_duplicates(subset=['nr_candidato', 'sg_partido', 'sg_uf'])

    eleito_mask = (
        ~r['ds_sit_tot_turno'].isin(['SUPLENTE']) &
        ~r['ds_sit_tot_turno'].str.startswith('N')
    )
    cadeiras = (
        r[eleito_mask]
        .groupby(['sg_partido', 'sg_uf'])
        .size()
        .reset_index(name='cadeiras')
        .assign(ano_eleicao=ano)
    )
    frames.append(cadeiras)

seats = pd.concat(frames, ignore_index=True)

# Verificação
check = seats.groupby('ano_eleicao')['cadeiras'].sum()
print('Cadeiras totais por ano (deve ser 513 cada):')
print(check)

## 2. NECr por lista × ano

Para 2018/2022 usa `rrd_df_novo.parquet` (já tem `prop_vr_receita_candidato`).  
Para 2014 calcula diretamente de `receitas.parquet`.

In [ ]:
# ── 2018 e 2022 via rrd ──────────────────────────────────────────────────────
rrd = pd.read_parquet('data/processed/rrd_df_novo.parquet')

def necr_from_rrd(rrd_sub):
    """Computa NECr por lista a partir de prop_vr_receita_candidato."""
    def agg(g):
        total = g['vr_receita_recursos_partidos'].sum()
        sum_sq = (g['prop_vr_receita_candidato'] ** 2).sum()
        return pd.Series({
            'NECr':     (1 / sum_sq) if sum_sq > 0 else np.nan,
            'vr_total': total,
            'n_cands':  len(g),
        })
    return (
        rrd_sub
        .groupby(['ano_eleicao', 'sg_partido', 'sg_uf'], observed=True)
        .apply(agg, include_groups=False)
        .reset_index()
    )

necr_1822 = necr_from_rrd(rrd[rrd.ano_eleicao.isin([2018, 2022])])
print(f'2018/2022: {len(necr_1822)} listas com recursos')

In [ ]:
# ── 2014 via receitas.parquet ─────────────────────────────────────────────────
rec = pd.read_parquet('data/processed/receitas.parquet')

# Janela: pré-eleição (antes de 2014-10-05)
r14 = rec[
    (rec.ano_eleicao == 2014) &
    (rec.ds_cargo == 'DEPUTADO FEDERAL') &
    rec['ds_origem_receita'].str.startswith('Recursos de partido') &
    (rec['dt_receita'] <= pd.Timestamp('2014-10-05'))
].copy()

# Total partido por lista
total_lista = (
    r14.groupby(['sg_partido', 'sg_uf'])['vr_receita']
    .sum()
    .reset_index()
    .rename(columns={'vr_receita': 'vr_total_lista'})
)

# Total por candidato dentro da lista
cand_rec = (
    r14.groupby(['sg_partido', 'sg_uf', 'nr_candidato'])['vr_receita']
    .sum()
    .reset_index()
    .rename(columns={'vr_receita': 'vr_cand'})
)

# Merge e proporção
cand_rec = cand_rec.merge(total_lista, on=['sg_partido', 'sg_uf'])
cand_rec['prop'] = cand_rec['vr_cand'] / cand_rec['vr_total_lista']

# NECr por lista
def agg14(g):
    sum_sq = (g['prop'] ** 2).sum()
    return pd.Series({
        'NECr':     (1 / sum_sq) if sum_sq > 0 else np.nan,
        'vr_total': g['vr_cand'].sum(),
        'n_cands':  len(g),
    })

necr_14 = (
    cand_rec
    .groupby(['sg_partido', 'sg_uf'])
    .apply(agg14, include_groups=False)
    .reset_index()
    .assign(ano_eleicao=2014)
)

print(f'2014: {len(necr_14)} listas com recursos de partido')

# Concatenar todos os anos
necr_all = pd.concat([necr_14, necr_1822], ignore_index=True)
necr_all['log_vr_total'] = np.log1p(necr_all['vr_total'])
print(f'Total: {len(necr_all)} listas')

## 3. Bancada anterior (B_prev)

In [ ]:
# bancada_partido_uf.csv tem 2014, 2018, 2022 (resultado da eleição de cada ano)
ban = pd.read_csv('data/processed/bancada_partido_uf.csv')

# 2010 bancada para B_prev da eleição de 2014
r10 = res[
    (res.ano_eleicao == 2010) &
    (res.ds_cargo == 'Deputado Federal') &
    (res.nr_turno == 1)
].drop_duplicates(subset=['nr_candidato', 'sg_partido', 'sg_uf'])

eleito10 = r10[
    ~r10['ds_sit_tot_turno'].isin(['SUPLENTE']) &
    ~r10['ds_sit_tot_turno'].str.startswith('N')
]
ban10 = (
    eleito10
    .groupby(['sg_partido', 'sg_uf'])
    .size()
    .reset_index(name='n_deputados')
    .assign(ano_eleicao=2010)
)
print(f'2010 bancada: {ban10["n_deputados"].sum()} cadeiras')

# Unir todas as bancadas e criar coluna de bancada_prev
# B_prev para a eleição do ano X é a bancada do ano X-4
ban_full = pd.concat([ban10, ban], ignore_index=True)

ano_prev_map = {2014: 2010, 2018: 2014, 2022: 2018}

frames_prev = []
for ano_alvo, ano_prev in ano_prev_map.items():
    sub = ban_full[ban_full.ano_eleicao == ano_prev][['sg_partido', 'sg_uf', 'n_deputados']].copy()
    sub['ano_eleicao'] = ano_alvo
    sub = sub.rename(columns={'n_deputados': 'bancada_prev'})
    frames_prev.append(sub)

bprev = pd.concat(frames_prev, ignore_index=True)
print(f'B_prev calculado para {len(bprev)} combinações partido×UF×ano')

## 4. Dataset analítico

In [ ]:
# Universo = listas que receberam recursos de partido
df = necr_all.merge(
    seats, on=['ano_eleicao', 'sg_partido', 'sg_uf'], how='left'
)
df['cadeiras'] = df['cadeiras'].fillna(0)

# Bancada anterior
df = df.merge(bprev, on=['ano_eleicao', 'sg_partido', 'sg_uf'], how='left')
df['bancada_prev'] = df['bancada_prev'].fillna(0)

# Magnitude
vagas = pd.read_parquet('data/processed/vagas_deputado_federal.parquet')
df = df.merge(
    vagas[['ano_eleicao', 'sg_uf', 'qt_vaga']],
    on=['ano_eleicao', 'sg_uf'], how='left'
)

# Remover listas sem NECr válido
df = df[df['NECr'].notna()].copy()

print('Dataset analítico:')
print(df.groupby('ano_eleicao').agg(
    n_listas=('NECr', 'count'),
    cadeiras_total=('cadeiras', 'sum'),
    NECr_mediana=('NECr', 'median'),
    bprev_mediana=('bancada_prev', 'median'),
).round(2))

## 5. Estatísticas descritivas comparativas

In [ ]:
# Correlações por ano
print('Correlações de Pearson (NECr × cadeiras, B_prev × cadeiras)\n')
for ano in ANOS_ALVO:
    sub = df[df.ano_eleicao == ano]
    r_necr = sub['NECr'].corr(sub['cadeiras'])
    r_bprev = sub['bancada_prev'].corr(sub['cadeiras'])
    r_logvr = sub['log_vr_total'].corr(sub['cadeiras'])
    r_necr_bprev = sub['NECr'].corr(sub['bancada_prev'])
    print(f'{ano}:  NECr×S = {r_necr:.3f}   B_prev×S = {r_bprev:.3f}   '
          f'logVR×S = {r_logvr:.3f}   NECr×B_prev = {r_necr_bprev:.3f}')

## 6. Modelos OLS por ano

Para cada ano, estima três especificações:
- (1) `S ~ NECr`
- (2) `S ~ B_prev`
- (3) `S ~ B_prev + NECr + log(VR_total)`

In [ ]:
from scipy import stats

resultados_modelos = []

for ano in ANOS_ALVO:
    sub = df[df.ano_eleicao == ano].dropna(subset=['NECr', 'bancada_prev', 'cadeiras', 'log_vr_total'])
    y = sub['cadeiras'].values

    specs = [
        ('(1) NECr',          ['NECr']),
        ('(2) B_prev',        ['bancada_prev']),
        ('(3) B_prev+NECr+VR', ['bancada_prev', 'NECr', 'log_vr_total']),
    ]

    for label, xvars in specs:
        X = sm.add_constant(sub[xvars].values)
        model = sm.OLS(y, X).fit(cov_type='HC3')
        yhat = model.fittedvalues
        mae  = mean_absolute_error(y, yhat)
        rmse = root_mean_squared_error(y, yhat)
        resultados_modelos.append({
            'Ano': ano,
            'Modelo': label,
            'N': len(sub),
            'R²': model.rsquared,
            'R²_adj': model.rsquared_adj,
            'MAE': mae,
            'RMSE': rmse,
        })

tab_modelos = pd.DataFrame(resultados_modelos)
print('Ajuste in-sample por ano e especificação\n')
print(tab_modelos.round(3).to_string(index=False))

In [ ]:
# Coeficientes detalhados modelo (3) por ano
print('Coeficientes do modelo completo (3) por ano\n')
for ano in ANOS_ALVO:
    sub = df[df.ano_eleicao == ano].dropna(subset=['NECr', 'bancada_prev', 'cadeiras', 'log_vr_total'])
    y = sub['cadeiras'].values
    X = sm.add_constant(sub[['bancada_prev', 'NECr', 'log_vr_total']].values)
    model = sm.OLS(y, X).fit(cov_type='HC3')
    coef = pd.Series(model.params, index=['const', 'bancada_prev', 'NECr', 'log_vr_total'])
    pval = pd.Series(model.pvalues, index=['const', 'bancada_prev', 'NECr', 'log_vr_total'])
    print(f'\n{ano} (N={len(sub)}, R²={model.rsquared:.3f}):')
    for v in coef.index:
        stars = '***' if pval[v] < 0.001 else '**' if pval[v] < 0.01 else '*' if pval[v] < 0.05 else ''
        print(f'  {v:<15} β={coef[v]:+.4f}  p={pval[v]:.3f} {stars}')

## 7. Validade cardinal: NECr vs. cadeiras conquistadas

Se NECr é um bom proxy de expectativas, `NECr ≈ cadeiras`. Verificamos a
calibração comparando médias e distribuições condicionadas a faixas de NECr.

In [ ]:
# Calibração cardinal: E[cadeiras | NECr ≈ k] = k ?
df['NECr_round'] = df['NECr'].round(0).clip(1, 20)

for ano in ANOS_ALVO:
    sub = df[(df.ano_eleicao == ano) & (df['NECr_round'] <= 15)]
    cal = (
        sub.groupby('NECr_round')
        .agg(
            n=('cadeiras', 'count'),
            cadeiras_media=('cadeiras', 'mean'),
            cadeiras_mediana=('cadeiras', 'median'),
        )
        .reset_index()
    )
    cal['razao_S_NECr'] = cal['cadeiras_media'] / cal['NECr_round']
    print(f'\n{ano} — Calibração cardinal (NECr arredondado × cadeiras médias):')
    print(cal[cal.n >= 3].round(2).to_string(index=False))

## 8. Validação cruzada temporal (out-of-sample)

Treina em um ano, prediz o outro. Avalia MAE e RMSE fora da amostra.

In [ ]:
oos_results = []

pairs = [(2018, 2022), (2022, 2018), (2014, 2018), (2018, 2014)]

for train_ano, test_ano in pairs:
    train = df[df.ano_eleicao == train_ano].dropna(
        subset=['NECr', 'bancada_prev', 'cadeiras', 'log_vr_total']
    )
    test = df[df.ano_eleicao == test_ano].dropna(
        subset=['NECr', 'bancada_prev', 'cadeiras', 'log_vr_total']
    )

    specs = [
        ('(1) NECr',           ['NECr']),
        ('(2) B_prev',         ['bancada_prev']),
        ('(3) B_prev+NECr+VR', ['bancada_prev', 'NECr', 'log_vr_total']),
    ]

    for label, xvars in specs:
        Xtr = sm.add_constant(train[xvars].values, has_constant='add')
        Xte = sm.add_constant(test[xvars].values,  has_constant='add')
        model = sm.OLS(train['cadeiras'].values, Xtr).fit()
        yhat = Xte @ model.params
        y    = test['cadeiras'].values
        mae  = mean_absolute_error(y, yhat)
        rmse = root_mean_squared_error(y, yhat)
        oos_results.append({
            'Treino→Teste': f'{train_ano}→{test_ano}',
            'Modelo': label,
            'N_treino': len(train),
            'N_teste':  len(test),
            'MAE_oos':  mae,
            'RMSE_oos': rmse,
        })

tab_oos = pd.DataFrame(oos_results)
print('Validação cruzada temporal (out-of-sample)\n')
print(tab_oos.round(3).to_string(index=False))

## 9. Validade ordinal: Spearman e concordância de ranking

In [ ]:
from scipy.stats import spearmanr, kendalltau

print('Correlação ordinal (Spearman ρ e Kendall τ) por ano\n')
for ano in ANOS_ALVO:
    sub = df[df.ano_eleicao == ano].dropna(subset=['NECr', 'bancada_prev', 'cadeiras'])
    s_necr,  _ = spearmanr(sub['NECr'],         sub['cadeiras'])
    s_bprev, _ = spearmanr(sub['bancada_prev'],  sub['cadeiras'])
    k_necr,  _ = kendalltau(sub['NECr'],         sub['cadeiras'])
    k_bprev, _ = kendalltau(sub['bancada_prev'],  sub['cadeiras'])
    print(f'{ano}:')
    print(f'  NECr   — Spearman ρ = {s_necr:.3f}, Kendall τ = {k_necr:.3f}')
    print(f'  B_prev — Spearman ρ = {s_bprev:.3f}, Kendall τ = {k_bprev:.3f}')

## 10. Síntese: tabela comparativa final

In [ ]:
print('=== SÍNTESE PREDITIVA DO NECr ===' + '\n')

# In-sample R² por modelo × ano
pivot = tab_modelos.pivot_table(
    index='Modelo', columns='Ano', values=['R²', 'MAE']
).round(3)
print('In-sample R² e MAE:\n')
print(pivot.to_string())

print('\n' + '-'*60)
print('Out-of-sample MAE:\n')
pivot_oos = tab_oos.pivot_table(
    index='Modelo', columns='Treino→Teste', values='MAE_oos'
).round(3)
print(pivot_oos.to_string())